# Differential gene expression between individuals who are non diabetic or individuals with type 2 diabetes

In [ ]:
# eval "$(conda shell.bash hook)"
# conda init
# conda activate /work/islet_cartography_scrna/scrna_cartography_deseq
# python -m ipykernel install --user --name scrna_cartography_dreampy --display-name "deseq"

In [ ]:
# Path and system utilities
import os                    # Operating system interface
import sys                   # System-specific parameters and functions
import glob                  # File pattern matching
from pathlib import Path     # Object-oriented filesystem paths
from pyhere import here      # Reproducible project paths
import gc

# Single-cell data handling
import anndata as ad            # Core data structure for single-cell data
import scanpy as sc

import matplotlib.pyplot as plt  # Plotting interface

# dream
from pydeseq2.dds import DeseqDataSet
from pydeseq2.default_inference import DefaultInference
from pydeseq2.ds import DeseqStats
import dreampy as dp

# Parallel processing
from joblib import Parallel, delayed, parallel_backend

# dataframes
import pandas as pd
import numpy as np
from collections import defaultdict

# Custom modules and functions
sys.path.append(str(here('scripts/misc')))  # Add custom script path to system
import misc as mi

#### Set up

In [ ]:
# Paths
base_dir = str(here('data/differential_genes_across_disease/'))
plot_dir = os.path.join(base_dir, 'plot') 
files_dir = os.path.join(base_dir, 'files') 
diffg_dir = os.path.join(base_dir, 'nd_t2d') 
tmp_dir = os.path.join(base_dir, 'tmp') 

anndata_dir = str(here('data/anndata/'))

mi.create_directories(os.path.join(base_dir, 'nd_t2d'))
mi.create_directories(os.path.join(plot_dir))
mi.create_directories(os.path.join(files_dir))
mi.create_directories(os.path.join(diffg_dir))
mi.create_directories(os.path.join(base_dir, 'tmp'))

#### Load

In [ ]:
adata = ad.read_h5ad(os.path.join(anndata_dir, "AH_combined.h5ad"))
duplicated_donors = pd.read_csv(str(here('data/duplicated_donors.csv')), sep = "\t")

#### Preprocess

In [ ]:
# Setup -----------------------------------------------------------------------------
anno_key   = "cell_type"
donor_key  = "ic_id_donor_overall"
dataset_key  = "ic_id_dataset"
disease_key = "disease_hba1c"
disease_id = "t2d"
ref_id = "nd"
comp = f"{disease_id}_vs_{ref_id}"
celltypes = adata.obs[anno_key].unique()
datasets = adata.obs[dataset_key].unique().tolist()
inference = DefaultInference(n_cpus=-1)

In [ ]:
# Create combined unique string keys
remove_keys = (
    duplicated_donors["ic_id_donor_overall"].astype(str)
    + "__"
    + duplicated_donors["ic_id_dataset"].astype(str)
)

obs_keys = (
    adata.obs["ic_id_donor_overall"].astype(str)
    + "__"
    + adata.obs["ic_id_dataset"].astype(str)
)

# Build mask and slice
mask_remove = obs_keys.isin(remove_keys)
adata = adata[~mask_remove].copy()

remaining_multi = adata.obs.groupby(donor_key)[dataset_key].nunique()
n_remaining_multi = (remaining_multi > 1).sum()
assert n_remaining_multi == 0, f"Error: {n_remaining_multi} donors still appear in multiple datasets!"
print("Filtering successful: all donors now in single dataset")
print("Number of donors: ", len(remaining_multi))
print(f"Removed {(mask_remove).sum()} observations")

#### Remove prediabetes samples

In [ ]:
pre_mask = adata.obs[disease_key] == "pre"
adata = adata[~pre_mask].copy()
assert "pre" not in adata.obs[disease_key].values, f"Error: 'pre' still present in {disease_key}!"
print("Successfully removed 'pre' from AnnData object.")

#### DEGs per dataset

In [ ]:
for celltype in celltypes:
    meta_results = []
    print(f"\n==============================")
    print(f"Running: {celltype}")
    print(f"==============================")

    mask = adata.obs[anno_key] == celltype
    adata_cell = adata[mask].copy()

    for dset in datasets:
        print(f"\n==============================")
        print(f"Running: {dset}")
        print(f"==============================")
        
        mask = adata_cell.obs[dataset_key] == dset
        ad_ds = adata_cell[mask].copy()
        cats = ad_ds.obs[disease_key].cat.categories.tolist()

        # If only one disease category, skip
        if len(cats) < 2:
            print(f"  Skipping {dset} - only one disease category")
            continue

        # add one assay
        ad_ds.obs['assay'] = 'my_assay'
        
        try:
            pb = dp.aggregate_pseudobulk(
                ad_ds, layer="counts", groupby=["assay", donor_key, disease_key]
            )
    
            # Find the highest min_cells threshold that gives ≥3 replicates for both groups
            pb_filtered = None
            min_cells_used = None
            
            for min_cells in [50, 20, 10, 5, 3]:
            
                pb_try = dp.filter_samples(
                    pb,
                    min_cells=min_cells,
                    min_samples=1,   # IMPORTANT
                )
            
                counts = pb_try.obs.groupby(disease_key)["assay"].count()
            
                print(
                    f"    min_cells={min_cells}: "
                    f"{disease_id}={counts.get(disease_id, 0)}, "
                    f"{ref_id}={counts.get(ref_id, 0)}"
                )
            
                if (counts.get(disease_id, 0) >= 3
                    and counts.get(ref_id, 0) >= 3):
                    
                    pb_filtered = pb_try
                    min_cells_used = min_cells
                    break
            
            if pb_filtered is None:
                raise ValueError(
                    "Could not get ≥3 replicates for both groups "
                    "at any threshold"
                )

            counts_df = pd.DataFrame(
                pb_filtered.X.toarray(),
                columns=pb_filtered.var_names,
                index=pb_filtered.obs_names,
            )
           
            metadata_df = pb_filtered.obs[[donor_key, disease_key]].copy()

            counts_df.index = counts_df.index.astype(str)
            metadata_df.index = metadata_df.index.astype(str)
            
            assert counts_df.index.equals(metadata_df.index)
            
            design_formula = f"~ {disease_key}"
    
            dds = DeseqDataSet(
                counts=counts_df,
                metadata=metadata_df,
                design=design_formula,
                inference=inference,
            )
            dds.deseq2()
    
            ds = DeseqStats(
                dds, contrast=(disease_key, disease_id, ref_id), inference=inference, quiet=True
            )
            ds.run_wald_test()
            ds.summary()

            n_donor = pd.concat([
                pb_filtered.obs[disease_key].value_counts().to_frame().T,
            ], axis=1)
            
            results = ds.results_df.copy()
            results["target"] = disease_id
            results["reference"] = ref_id
            results["comparison"] = comp
            results["n_donors"]   = dds.shape[0]
            results["dataset"]    = dset
            results["min_cells"]  = min_cells_used
            results[n_donor.columns.tolist()] = n_donor.iloc[0]

            meta_results.append(results)
            results.to_csv(
                os.path.join(tmp_dir, f"{celltype}_{comp}_{dset}.csv"),
                index=True, index_label="gene_symbol",
            )
    
        except Exception as e:
            print(f"  Skipping {dset}: {e}")
        
    # Combine all results for this cluster
    if len(meta_results) > 0:
        meta_df = pd.concat(meta_results, ignore_index=False)
        meta_df.to_csv(
            os.path.join(diffg_dir, f"{celltype}.csv"), 
            index=True, 
            index_label="gene_symbol"
        )
    else:
        print(f"Skipping entire cluster {celltype} - all datasets failed") 